# Transformer Transcript Visualizer
### Interactive visualization of transformer attention patterns and token processing

This notebook creates interactive visualizations similar to Transformer Explainer for analyzing how transformers process text transcripts.

In [ ]:
# Install required packages
!pip install -q transformers torch matplotlib plotly bertviz numpy pandas seaborn ipywidgets

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from bertviz import head_view, model_view
import ipywidgets as widgets
from IPython.display import display, HTML
import pandas as pd
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load GPT-2 Model
We'll load a GPT-2 model and configure it to output attention weights.

In [ ]:
# Load GPT-2 model and tokenizer
model_name = "gpt2"  # You can also use "gpt2-medium", "gpt2-large", etc.
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name, output_attentions=True)
model.eval()

print(f"✓ Loaded {model_name} model")
print(f"  - Layers: {model.config.n_layer}")
print(f"  - Attention heads: {model.config.n_head}")
print(f"  - Hidden size: {model.config.n_embd}")

## 2. Process Text and Extract Attention

In [ ]:
def process_text(text, max_length=100):
    """
    Process text through the model and extract attention patterns.
    
    Args:
        text: Input text to analyze
        max_length: Maximum sequence length
    
    Returns:
        Dictionary containing tokens, attention weights, and predictions
    """
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    
    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Extract components
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    attentions = outputs.attentions  # Tuple of (num_layers, batch, num_heads, seq_len, seq_len)
    logits = outputs.logits
    
    # Get next token predictions
    next_token_logits = logits[0, -1, :]
    top_k = 10
    top_tokens = torch.topk(next_token_logits, top_k)
    predicted_tokens = [(tokenizer.decode([idx.item()]), prob.item()) 
                       for idx, prob in zip(top_tokens.indices, torch.softmax(top_tokens.values, dim=0))]
    
    return {
        'tokens': tokens,
        'attentions': attentions,
        'predictions': predicted_tokens,
        'num_layers': len(attentions),
        'num_heads': attentions[0].shape[1]
    }

## 3. Attention Visualization Functions

In [ ]:
def visualize_attention_heatmap(attention_weights, tokens, layer=0, head=0, title=None):
    """
    Create an interactive heatmap of attention weights.
    """
    # Get attention for specific layer and head
    attn = attention_weights[layer][0, head].detach().numpy()
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=attn,
        x=tokens,
        y=tokens,
        colorscale='Viridis',
        hoverongaps=False,
        hovertemplate='From: %{y}<br>To: %{x}<br>Attention: %{z:.3f}<extra></extra>'
    ))
    
    title = title or f'Attention Pattern - Layer {layer}, Head {head}'
    fig.update_layout(
        title=title,
        xaxis_title='To Token',
        yaxis_title='From Token',
        width=800,
        height=800,
        xaxis={'side': 'bottom'},
        yaxis={'autorange': 'reversed'}
    )
    
    return fig


def visualize_attention_flow(attention_weights, tokens, layer=0, source_token_idx=0):
    """
    Visualize how a specific token attends to others across all heads.
    """
    num_heads = attention_weights[layer].shape[1]
    seq_len = len(tokens)
    
    # Extract attention from source token to all others for each head
    attn_data = []
    for head in range(num_heads):
        attn = attention_weights[layer][0, head, source_token_idx].detach().numpy()
        for i, (token, weight) in enumerate(zip(tokens, attn)):
            attn_data.append({
                'Head': f'H{head}',
                'Token': f'{i}: {token}',
                'Attention': weight,
                'Token_Idx': i
            })
    
    df = pd.DataFrame(attn_data)
    
    # Create grouped bar chart
    fig = px.bar(df, x='Token', y='Attention', color='Head',
                 barmode='group',
                 title=f'Attention from Token "{tokens[source_token_idx]}" (Layer {layer})',
                 labels={'Attention': 'Attention Weight'})
    
    fig.update_layout(
        width=1200,
        height=500,
        xaxis_tickangle=-45
    )
    
    return fig


def visualize_layer_aggregation(attention_weights, tokens, token_idx=0):
    """
    Show how attention from a token changes across layers.
    """
    num_layers = len(attention_weights)
    num_heads = attention_weights[0].shape[1]
    
    # Average attention across heads for each layer
    layer_attns = []
    for layer in range(num_layers):
        # Average across all heads
        avg_attn = attention_weights[layer][0, :, token_idx, :].mean(dim=0).detach().numpy()
        layer_attns.append(avg_attn)
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=layer_attns,
        x=tokens,
        y=[f'Layer {i}' for i in range(num_layers)],
        colorscale='Plasma',
        hovertemplate='Layer: %{y}<br>Token: %{x}<br>Avg Attention: %{z:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'Attention Evolution Across Layers (from "{tokens[token_idx]}")',
        xaxis_title='To Token',
        yaxis_title='Layer',
        width=1200,
        height=600,
        yaxis={'autorange': 'reversed'}
    )
    
    return fig


def visualize_predictions(predictions):
    """
    Visualize top predicted next tokens.
    """
    tokens, probs = zip(*predictions)
    
    fig = go.Figure(data=[
        go.Bar(
            x=list(probs),
            y=list(tokens),
            orientation='h',
            marker=dict(
                color=list(probs),
                colorscale='Blues',
                showscale=False
            ),
            text=[f'{p:.1%}' for p in probs],
            textposition='auto',
        )
    ])
    
    fig.update_layout(
        title='Top 10 Predicted Next Tokens',
        xaxis_title='Probability',
        yaxis_title='Token',
        width=600,
        height=400,
        yaxis={'autorange': 'reversed'}
    )
    
    return fig


def create_attention_graph(attention_weights, tokens, layer=0, head=0, threshold=0.1):
    """
    Create a network graph showing attention connections.
    """
    attn = attention_weights[layer][0, head].detach().numpy()
    
    # Filter weak connections
    strong_connections = np.where(attn > threshold)
    
    # Create edge trace
    edge_traces = []
    
    # Position tokens in a circle
    n = len(tokens)
    angles = np.linspace(0, 2*np.pi, n, endpoint=False)
    x_pos = np.cos(angles)
    y_pos = np.sin(angles)
    
    for i, j in zip(*strong_connections):
        if i != j:  # Skip self-attention for clarity
            edge_trace = go.Scatter(
                x=[x_pos[i], x_pos[j], None],
                y=[y_pos[i], y_pos[j], None],
                mode='lines',
                line=dict(width=attn[i,j]*5, color=f'rgba(100,100,250,{attn[i,j]})'),
                hoverinfo='none',
                showlegend=False
            )
            edge_traces.append(edge_trace)
    
    # Create node trace
    node_trace = go.Scatter(
        x=x_pos,
        y=y_pos,
        mode='markers+text',
        marker=dict(size=20, color='lightblue', line=dict(width=2, color='darkblue')),
        text=tokens,
        textposition='top center',
        hoverinfo='text',
        showlegend=False
    )
    
    fig = go.Figure(data=edge_traces + [node_trace])
    fig.update_layout(
        title=f'Attention Graph - Layer {layer}, Head {head} (threshold={threshold})',
        showlegend=False,
        width=800,
        height=800,
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
    )
    
    return fig

## 4. Interactive Widget Interface

In [ ]:
class TransformerVisualizer:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.result = None
        
        # Create widgets
        self.text_input = widgets.Textarea(
            value='The transformer model processes text by',
            description='Input:',
            layout=widgets.Layout(width='100%', height='100px')
        )
        
        self.process_button = widgets.Button(
            description='Analyze',
            button_style='primary',
            icon='search'
        )
        
        self.layer_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=11,  # Will update based on model
            description='Layer:',
            continuous_update=False
        )
        
        self.head_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=11,  # Will update based on model
            description='Head:',
            continuous_update=False
        )
        
        self.token_dropdown = widgets.Dropdown(
            options=[],
            description='Token:'
        )
        
        self.viz_type = widgets.Dropdown(
            options=[
                ('Attention Heatmap', 'heatmap'),
                ('Attention Flow', 'flow'),
                ('Layer Evolution', 'evolution'),
                ('Attention Graph', 'graph'),
                ('Predictions', 'predictions')
            ],
            description='Viz Type:',
            value='heatmap'
        )
        
        self.output = widgets.Output()
        
        # Set up event handlers
        self.process_button.on_click(self.on_process_click)
        self.layer_slider.observe(self.on_param_change, 'value')
        self.head_slider.observe(self.on_param_change, 'value')
        self.token_dropdown.observe(self.on_param_change, 'value')
        self.viz_type.observe(self.on_param_change, 'value')
    
    def on_process_click(self, b):
        with self.output:
            self.output.clear_output()
            print("Processing text...")
            
            # Process the text
            self.result = process_text(self.text_input.value)
            
            # Update widget ranges
            self.layer_slider.max = self.result['num_layers'] - 1
            self.head_slider.max = self.result['num_heads'] - 1
            self.token_dropdown.options = [(f"{i}: {t}", i) for i, t in enumerate(self.result['tokens'])]
            self.token_dropdown.value = 0
            
            print(f"✓ Processed {len(self.result['tokens'])} tokens")
            self.update_visualization()
    
    def on_param_change(self, change):
        if self.result is not None:
            self.update_visualization()
    
    def update_visualization(self):
        with self.output:
            self.output.clear_output(wait=True)
            
            viz_type = self.viz_type.value
            
            if viz_type == 'heatmap':
                fig = visualize_attention_heatmap(
                    self.result['attentions'],
                    self.result['tokens'],
                    layer=self.layer_slider.value,
                    head=self.head_slider.value
                )
            elif viz_type == 'flow':
                fig = visualize_attention_flow(
                    self.result['attentions'],
                    self.result['tokens'],
                    layer=self.layer_slider.value,
                    source_token_idx=self.token_dropdown.value
                )
            elif viz_type == 'evolution':
                fig = visualize_layer_aggregation(
                    self.result['attentions'],
                    self.result['tokens'],
                    token_idx=self.token_dropdown.value
                )
            elif viz_type == 'graph':
                fig = create_attention_graph(
                    self.result['attentions'],
                    self.result['tokens'],
                    layer=self.layer_slider.value,
                    head=self.head_slider.value,
                    threshold=0.1
                )
            else:  # predictions
                fig = visualize_predictions(self.result['predictions'])
            
            fig.show()
    
    def display(self):
        display(HTML("<h2>🔍 Transformer Transcript Visualizer</h2>"))
        display(self.text_input)
        display(self.process_button)
        display(HTML("<hr>"))
        display(widgets.HBox([self.viz_type, self.layer_slider, self.head_slider, self.token_dropdown]))
        display(self.output)

## 5. Launch Interactive Visualizer

In [ ]:
# Create and display the visualizer
visualizer = TransformerVisualizer(model, tokenizer)
visualizer.display()

## 6. Quick Analysis Examples

Below are some pre-configured analyses you can run directly.

In [ ]:
# Example 1: Analyze a transcript snippet
example_text = """The meeting started at 9 AM. John presented the quarterly results, 
showing a 15% increase in revenue. Sarah raised concerns about the budget."""

result = process_text(example_text)

print(f"Tokens: {result['tokens']}")
print(f"\nTop predictions for next token:")
for token, prob in result['predictions'][:5]:
    print(f"  {token:20s} {prob:.1%}")

In [ ]:
# Example 2: Compare attention patterns across layers
fig = visualize_layer_aggregation(result['attentions'], result['tokens'], token_idx=5)
fig.show()

In [ ]:
# Example 3: Attention heatmap for specific layer and head
fig = visualize_attention_heatmap(result['attentions'], result['tokens'], layer=6, head=0)
fig.show()

In [ ]:
# Example 4: Attention graph visualization
fig = create_attention_graph(result['attentions'], result['tokens'], layer=8, head=3, threshold=0.15)
fig.show()

## 7. Export Attention Data

Extract attention weights for further analysis.

In [ ]:
def export_attention_matrix(result, layer=0, head=0):
    """
    Export attention weights as a pandas DataFrame.
    """
    attn = result['attentions'][layer][0, head].detach().numpy()
    tokens = result['tokens']
    
    df = pd.DataFrame(attn, columns=tokens, index=tokens)
    return df

# Example usage
attn_df = export_attention_matrix(result, layer=6, head=0)
print("Attention matrix shape:", attn_df.shape)
attn_df.head(10)

## 8. Analyze Multiple Texts

Batch process and compare multiple transcript segments.

In [ ]:
def compare_attention_patterns(texts, layer=6, head=0):
    """
    Compare attention patterns across multiple texts.
    """
    results = [process_text(text) for text in texts]
    
    fig = make_subplots(
        rows=1, cols=len(texts),
        subplot_titles=[f"Text {i+1}" for i in range(len(texts))]
    )
    
    for i, result in enumerate(results):
        attn = result['attentions'][layer][0, head].detach().numpy()
        
        fig.add_trace(
            go.Heatmap(
                z=attn,
                x=result['tokens'],
                y=result['tokens'],
                colorscale='Viridis',
                showscale=(i == len(texts)-1)
            ),
            row=1, col=i+1
        )
    
    fig.update_layout(
        title=f'Attention Pattern Comparison - Layer {layer}, Head {head}',
        width=400*len(texts),
        height=400
    )
    
    return fig

# Example: Compare different sentence structures
comparison_texts = [
    "The cat sat on the mat.",
    "On the mat sat the cat.",
    "The mat had a cat sitting on it."
]

fig = compare_attention_patterns(comparison_texts, layer=6, head=0)
fig.show()

## 9. Advanced: BertViz Integration

Use BertViz for additional interactive visualizations.

In [ ]:
from bertviz import head_view, model_view
from transformers import GPT2Model

# Note: BertViz works best with encoder models, but can still provide insights for GPT-2
def show_bertviz(text):
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs, output_attentions=True)
    
    attention = outputs.attentions
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    
    # Show head view
    head_view(attention, tokens)

# Uncomment to use:
# show_bertviz("The transformer architecture revolutionized natural language processing.")

## 10. Tips for Using This Notebook

1. **Start with the Interactive Visualizer (Section 5)**: Use the widget interface to explore different texts interactively

2. **Attention Patterns to Look For**:
   - **Diagonal patterns**: Model attending to recent tokens (local context)
   - **Vertical lines**: Specific tokens getting high attention (important words)
   - **Block patterns**: Phrase-level attention

3. **Layer Insights**:
   - **Early layers**: Often capture syntax and local patterns
   - **Middle layers**: Build semantic representations
   - **Late layers**: Task-specific processing and prediction

4. **Head Specialization**: Different heads often learn different patterns (e.g., syntactic vs semantic)

5. **Try Different Texts**:
   - Transcripts with dialogue
   - Technical content
   - Narrative text
   - Compare how attention differs

---

### 📚 Additional Resources
- [Transformer Explainer](https://poloclub.github.io/transformer-explainer/)
- [BertViz Documentation](https://github.com/jessevig/bertviz)
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
- [Attention Is All You Need (Paper)](https://arxiv.org/abs/1706.03762)